In [8]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.interpolate import interp1d
from collections import Counter
import statsmodels.api as sm
from statsmodels.formula.api import ols
import pandas as pd
import os

In [4]:
def load_wesad_data(directory):
    data = {}
    for participant_id in os.listdir(directory):
        participant_path = os.path.join(directory, participant_id)
        if os.path.isdir(participant_path):
            participant_data = {}
            for file_name in os.listdir(participant_path):
                file_path = os.path.join(participant_path, file_name)
                if file_name.endswith('.pkl'):
                    participant_data[file_name.split('.')[0]] = pd.read_pickle(file_path)
            data[participant_id] = participant_data
    return data

wesad_data = load_wesad_data('C:/Users/rusha/Desktop/Uni_Freiburg_Notes/MDD/WESAD')

In [17]:
def extract_and_process_data(subject_id):
    subject_data = wesad_data[subject_id]
    subject_details = subject_data[subject_id]
    chest_data = subject_details['signal']['chest']
    labels = subject_details['label']
    
    # Flatten and reshape each signal to ensure proper concatenation
    ecg = chest_data['ECG'].flatten().reshape(-1, 1)
    emg = chest_data['EMG'].flatten().reshape(-1, 1)
    eda = chest_data['EDA'].flatten().reshape(-1, 1)
    temp = chest_data['Temp'].flatten().reshape(-1, 1)
    resp = chest_data['Resp'].flatten().reshape(-1, 1)
    
    # Print the shapes of all features to ensure they match
    print(f"ECG shape: {ecg.shape}")
    print(f"EMG shape: {emg.shape}")
    print(f"EDA shape: {eda.shape}")
    print(f"TEMP shape: {temp.shape}")
    print(f"RESP shape: {resp.shape}")
    
    # Combine the features into a single array
    features = np.hstack((
        ecg,
        emg,
        eda,
        temp,
        resp
    ))

    return features, labels

# Extract and process data from subjects S10 and S11
features_s10, labels_s10 = extract_and_process_data('S10')
features_s11, labels_s11 = extract_and_process_data('S11')

# Combine features and labels from both subjects
combined_features = np.vstack((features_s10, features_s11))
combined_labels = np.hstack((labels_s10, labels_s11))

l = pd.DataFrame(np.hstack((combined_features, combined_labels.reshape(-1, 1))))

ECG shape: (3847200, 1)
EMG shape: (3847200, 1)
EDA shape: (3847200, 1)
TEMP shape: (3847200, 1)
RESP shape: (3847200, 1)
ECG shape: (3663100, 1)
EMG shape: (3663100, 1)
EDA shape: (3663100, 1)
TEMP shape: (3663100, 1)
RESP shape: (3663100, 1)


In [18]:
l = l[l[5] != 5]
l = l[l[5] != 6]
l = l[l[5] != 7]

combined_features = l.loc[:,:4]
combined_labels = l.loc[:, 5]

In [35]:
if not isinstance(combined_features, pd.DataFrame):
    X = pd.DataFrame(combined_features)

if not isinstance(combined_labels, pd.Series):
    y = pd.Series(combined_labels, name='label')

X.columns = ['ECG', 'EMG', 'EDA', 'temp', 'resp']

# Combine X and y into a single DataFrame
data = pd.concat([X, y], axis=1)
data = data.sample(n=100000, random_state=42)
# Print the column names to debug
print(data.columns)

# Perform ANOVA for each feature in X
anova_results = {}
for column in X.columns:
    try:
        model = ols(f'{column} ~ C(label)', data=data).fit()
        anova_table = sm.stats.anova_lm(model, typ=2)
        anova_results[column] = anova_table
    except Exception as e:
        print(f"Error processing feature {column}: {e}")

# Print ANOVA results for each feature
for feature, anova_table in anova_results.items():
    print(f"ANOVA results for feature: {feature}")
    print(anova_table)
    print("\n")

Index(['ECG', 'EMG', 'EDA', 'temp', 'resp', 'label'], dtype='object')
ANOVA results for feature: ECG
               sum_sq       df         F    PR(>F)
C(label)     0.119103      7.0  0.315146  0.947561
Residual  5398.555385  99992.0       NaN       NaN


ANOVA results for feature: EMG
             sum_sq       df         F    PR(>F)
C(label)   0.001132      7.0  1.136416  0.336597
Residual  14.224718  99992.0       NaN       NaN


ANOVA results for feature: EDA
                 sum_sq       df           F         PR(>F)
C(label)    9057.395907      7.0  186.861558  2.048418e-276
Residual  692389.704029  99992.0         NaN            NaN


ANOVA results for feature: temp
                sum_sq       df            F  PR(>F)
C(label)   5261.093461      7.0  6333.983458     0.0
Residual  11864.960786  99992.0          NaN     NaN


ANOVA results for feature: resp
                sum_sq       df        F    PR(>F)
C(label)  1.056980e+02      7.0  0.87517  0.525098
Residual  1.725208e+06  